In [64]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from scipy.stats import zscore
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [65]:
dataset = pd.read_csv('heart.csv')
dataset.drop(columns = ['Age', 'Sex'], inplace = True)

In [66]:
dataset.head()

,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,ATA,130,283,0,ST,98,N,0.0,Up,0
3,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [67]:
print(dataset.shape)

(918, 10)


In [68]:
numeric_cols = dataset.select_dtypes(include = np.number).columns
z_scores = np.abs(zscore(dataset[numeric_cols], nan_policy = 'omit'))
dataset = dataset[(z_scores <= 3).all(axis = 1)]

In [69]:
dataset.head()

,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,ATA,130,283,0,ST,98,N,0.0,Up,0
3,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [70]:
print(dataset.shape)

(899, 10)


In [71]:
lbe_chestpaintype = LabelEncoder()
ohe_chestpaintype = OneHotEncoder(sparse_output = False)
lbe_restingecg = LabelEncoder()
ohe_restingecg = OneHotEncoder(sparse_output = False)
lbe_exerciseangina = LabelEncoder()
ohe_exerciseangina = OneHotEncoder(sparse_output = False)
lbe_stslope = LabelEncoder()
ohe_stslope = OneHotEncoder(sparse_output = False)

In [72]:
cpt = lbe_chestpaintype.fit_transform(dataset.ChestPainType)
cpt = ohe_chestpaintype.fit_transform(cpt.reshape(-1, 1))
recg = lbe_restingecg.fit_transform(dataset.RestingECG)
recg = ohe_restingecg.fit_transform(recg.reshape(-1, 1))
eang = lbe_exerciseangina.fit_transform(dataset.ExerciseAngina)
eang = ohe_exerciseangina.fit_transform(eang.reshape(-1, 1))
sts = lbe_stslope.fit_transform(dataset.ST_Slope)
sts = ohe_stslope.fit_transform(sts.reshape(-1, 1))

In [73]:
cpt.shape, recg.shape, eang.shape, sts.shape

((899, 4), (899, 3), (899, 2), (899, 3))

In [74]:
cpt_df = pd.DataFrame(
    cpt.toarray() if hasattr(cpt, "toarray") else cpt,
    columns = lbe_chestpaintype.classes_,
    index = dataset.index
).add_prefix("ChestPainType_")
recg_df = pd.DataFrame(
    recg.toarray() if hasattr(recg, "toarray") else recg,
    columns = lbe_restingecg.classes_,
    index = dataset.index
).add_prefix("RestingECG_")
eang_df = pd.DataFrame(
    eang.toarray() if hasattr(eang, "toarray") else eang,
    columns = lbe_exerciseangina.classes_,
    index = dataset.index
).add_prefix("ExerciseAngina_")
sts_df = pd.DataFrame(
    sts.toarray() if hasattr(sts, "toarray") else sts,
    columns = lbe_stslope.classes_,
    index = dataset.index
).add_prefix("STSlope_")

In [75]:
dataset = dataset.drop(columns = ["ChestPainType", 'RestingECG', 'ExerciseAngina', 'ST_Slope'])
dataset = pd.concat([dataset, cpt_df, recg_df, eang_df, sts_df], axis = 1)

In [76]:
X = dataset.drop(columns = ["HeartDisease"]).to_numpy()
Y = dataset.HeartDisease.to_numpy()

In [77]:
X, Y

(array([[140., 289.,   0., ...,   0.,   0.,   1.],
        [160., 180.,   0., ...,   0.,   1.,   0.],
        [130., 283.,   0., ...,   0.,   0.,   1.],
        ...,
        [130., 131.,   0., ...,   0.,   1.,   0.],
        [130., 236.,   0., ...,   0.,   1.,   0.],
        [138., 175.,   0., ...,   0.,   0.,   1.]], shape=(899, 17)),
 array([0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
        1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0,
        0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1,
        0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
        0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0,
        1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,

In [78]:
sc = StandardScaler()
X = sc.fit_transform(X)

In [79]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2)

In [80]:
x_train.shape, x_test.shape

((719, 17), (180, 17))

In [82]:
lre = LogisticRegression()
lre.fit(x_train, y_train)
lre.score(x_test, y_test)

0.8722222222222222

In [83]:
pca = PCA(0.95)
X = pca.fit_transform(X)

In [85]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2)

In [86]:
lre = LogisticRegression()
lre.fit(x_train, y_train)
lre.score(x_test, y_test)

0.8555555555555555